# Data Vizualization

## Loading Packages

In [1]:
import requests
import numpy as np
import pandas as pd

import glob
import plotly.express as px
import folium

import os

from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

import plotly.graph_objects as go

In [2]:
output_dir = "../data"

In [3]:
weather_daily = pd.read_csv('../data/JC/jersey_weather_2025.csv')
citibike_df = pd.read_csv('../data/JC/JC2025_Enriched.csv')

## Monthly Rides

In [4]:
monthly_rides = (citibike_df
                 .groupby('month', as_index=False)
                 .agg(number_of_rides = ('ride_id','count'))
                 )
monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50477
2,2025-02,45131
3,2025-03,73124
4,2025-04,81295
5,2025-05,92880
6,2025-06,96736
7,2025-07,107374
8,2025-08,108001
9,2025-09,115580


### Number of Citi Bike Rides per Month

In [5]:
fig = px.bar(
    data_frame=monthly_rides,
    x='month',
    y='number_of_rides',
    title='Number of Citi Bike Rides per Month'   
)

fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Number of Rides'
)

fig.show()

In [6]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,day_of_week,hour,month_number,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,...,-74.051789,member,6.192900,2025-01-16,2025-01,January,Thursday,17,1,Winter
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,...,-74.078900,member,11.461350,2025-01-31,2025-01,January,Friday,6,1,Winter
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,21.377617,2025-01-09,2025-01,January,Thursday,16,1,Winter
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,22.934333,2025-01-21,2025-01,January,Tuesday,16,1,Winter
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,25.822100,2025-01-30,2025-01,January,Thursday,16,1,Winter


## Day of Week Rides

In [7]:
cats = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

citibike_df['day_of_week'] = pd.Categorical(citibike_df['day_of_week'], categories=cats, ordered=True)

day_of_week_rides = (citibike_df
                     .groupby('day_of_week', as_index=False)
                     .agg(number_of_rides=('ride_id', 'count'))
                     .sort_values('day_of_week') 
                    )

max_idx = day_of_week_rides['number_of_rides'].idxmax()
min_idx = day_of_week_rides['number_of_rides'].idxmin()

highest_day = day_of_week_rides.loc[max_idx]
lowest_day = day_of_week_rides.loc[min_idx]

print(day_of_week_rides)
print(f"Highest Day: {highest_day['day_of_week']} ({highest_day['number_of_rides']} rides)")
print(f"Lowest Day: {lowest_day['day_of_week']} ({lowest_day['number_of_rides']} rides)")

  day_of_week  number_of_rides
0      Monday           134590
1     Tuesday           148385
2   Wednesday           148050
3    Thursday           150697
4      Friday           157044
5    Saturday           140267
6      Sunday           119248
Highest Day: Friday (157044 rides)
Lowest Day: Sunday (119248 rides)


### Number of Citi Bike Rides per Day of Week

In [8]:
max_val = day_of_week_rides['number_of_rides'].max()
min_val = day_of_week_rides['number_of_rides'].min()

fig = px.bar(
    data_frame=day_of_week_rides,
    x='day_of_week',
    y='number_of_rides',
    title='Number of Citi Bike Rides per Day of Week',
    color=['green' if v == max_val else 'crimson' if v == min_val else 'lightgray' for v in day_of_week_rides['number_of_rides']],
)   


fig.update_layout(
    xaxis_title='Day of Week',
    yaxis_title='Number of Rides',
    showlegend=False
)

fig.show()

## Rides per Season of Year

In [9]:
cats = ['Winter', 'Spring', 'Summer', 'Autumn']

citibike_df['season'] = pd.Categorical(citibike_df['season'], categories=cats, ordered=True)

season_rides = (citibike_df
                     .groupby('season', as_index=False)
                     .agg(number_of_rides=('ride_id', 'count'))
                     .sort_values('season') 
                    )

season_rides

,season,number_of_rides
0,Winter,143472
1,Spring,247299
2,Summer,312111
3,Autumn,295399


### Number of Citi Bike Rides per Season of Year

In [10]:
fig = px.bar(
    data_frame=season_rides,
    x='season',
    y='number_of_rides',
    title='Number of Citi Bike Rides per Season of Year'
)   

fig.update_layout(
    xaxis_title='Season of Year',
    yaxis_title='Number of Rides'
)

fig.show()

## Rides per Time of Day

In [11]:
def assign_hour(hour):
    if hour in [6,7,8,9,10,11]:
        return 'Morning'
    elif hour in[12,13,14,15,16,17,18]:
        return 'DayTime'
    elif hour in [19,20,21,22,23,24]:
        return 'Evening'
    else:
        return 'Night'

In [12]:
citibike_df['time_of_day'] = citibike_df['hour'].apply(assign_hour)
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,member_casual,ride_duration_min,date,month,month_name,day_of_week,hour,month_number,season,time_of_day
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,...,member,6.192900,2025-01-16,2025-01,January,Thursday,17,1,Winter,DayTime
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,...,member,11.461350,2025-01-31,2025-01,January,Friday,6,1,Winter,Morning
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,member,21.377617,2025-01-09,2025-01,January,Thursday,16,1,Winter,DayTime
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,member,22.934333,2025-01-21,2025-01,January,Tuesday,16,1,Winter,DayTime
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,member,25.822100,2025-01-30,2025-01,January,Thursday,16,1,Winter,DayTime


In [13]:
cats = ['Morning', 'DayTime', 'Evening', 'Night']

citibike_df['time_of_day'] = pd.Categorical(citibike_df['time_of_day'], categories=cats, ordered=True)
time_of_day_rides = (citibike_df
                     .groupby('time_of_day', as_index=False)
                     .agg(number_of_rides=('ride_id', 'count'))
                     .sort_values('time_of_day') 
                    )

time_of_day_rides

,time_of_day,number_of_rides
0,Morning,286757
1,DayTime,482342
2,Evening,190522
3,Night,38660


### Number of Citi Bike Rides per Time of Day

In [14]:
fig = px.line(
    data_frame=time_of_day_rides,
    x='time_of_day',
    y='number_of_rides',
    title='Number of Citi Bike Rides per Time of Day'
)   

fig.update_layout(
    xaxis_title='Time of Day',
    yaxis_title='Number of Rides'
)

fig.show()

## Top Start Stations

In [15]:
top_start_station = (citibike_df
               .groupby('start_station_name', as_index=False)
               .agg(number_of_departures=('ride_id', 'count'))
               .sort_values('number_of_departures', ascending=False)
               .head(10))

top_start_station

,start_station_name,number_of_departures
52,Grove St PATH,44984
58,Hoboken Terminal - Hudson St & Hudson Pl,25879
53,Hamilton Park,22232
95,River St & Newark St,21383
86,Newport PATH,20641
18,Bergen Ave & Sip Ave,20370
44,Exchange Pl,19982
0,11 St & Washington St,19469
94,River St & 1 St,19125
87,Newport Pkwy,18709


### Top 10 Start Stations by Number of Departures

In [16]:
fig = px.bar(
    top_start_station.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

## Top End Stations

In [17]:
top_end_station = (citibike_df
               .groupby('end_station_name', as_index=False)
               .agg(number_of_arrivals=('ride_id', 'count'))
               .sort_values('number_of_arrivals', ascending=False)
               .head(10))

top_end_station

,end_station_name,number_of_arrivals
232,Grove St PATH,47744
241,Hoboken Terminal - Hudson St & Hudson Pl,26638
233,Hamilton Park,22347
347,River St & Newark St,22113
317,Newport PATH,20698
73,Bergen Ave & Sip Ave,20357
207,Exchange Pl,20142
7,11 St & Washington St,19501
318,Newport Pkwy,18704
346,River St & 1 St,18515


### Top 10 End Stations by Number of Arrivals

In [18]:
fig = px.bar(
    data_frame=top_end_station,
    x='number_of_arrivals',
    y='end_station_name',
    orientation='h',
    title = 'Top 10 End Stations by Number of Arrivals',
    text_auto=True
)

fig.update_layout(
    xaxis_title='Number of Arrivals',
    yaxis_title='End Station'
)

fig.show()

## Top 10 Stations 

In [19]:
top_stations = pd.concat([
    top_start_station.rename(columns={
                             'start_station_name': 'station',
                             'number_of_departures': 'number_of_arrivals_and_departures'}),
    top_end_station.rename(columns={
                           'end_station_name': 'station',
                           'number_of_arrivals': 'number_of_arrivals_and_departures'})
])

top_stations.head()

,station,number_of_arrivals_and_departures
52,Grove St PATH,44984
58,Hoboken Terminal - Hudson St & Hudson Pl,25879
53,Hamilton Park,22232
95,River St & Newark St,21383
86,Newport PATH,20641


### Top 10 Stations by Number of Departures and Arrivals

In [20]:
fig = px.bar(
    data_frame=top_stations,
    x='number_of_arrivals_and_departures',
    y='station',
    orientation='h',
    title = 'Top 10 Stations by Number of Departures and Arrivals',
    text_auto=True
)

fig.update_layout(
    xaxis_title='Number of Arrivals and Departures',
    yaxis_title='Station'
)

fig.show()

## Merge with Weather Data

In [21]:
daily_rides = (citibike_df
               .groupby('date', as_index=False)
               .agg(number_of_daily_rides = ('ride_id', 'count'))
               )

daily_rides.head()

,date,number_of_daily_rides
0,2024-12-31,2
1,2025-01-01,1174
2,2025-01-02,1709
3,2025-01-03,1764
4,2025-01-04,1336


In [22]:
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   date                 365 non-null    str    
 1   temperature_2m_max   365 non-null    float64
 2   temperature_2m_min   365 non-null    float64
 3   temperature_2m_mean  365 non-null    float64
 4   precipitation_sum    365 non-null    float64
 5   rain_sum             365 non-null    float64
 6   snowfall_sum         365 non-null    float64
 7   wind_speed_10m_max   365 non-null    float64
dtypes: float64(7), str(1)
memory usage: 26.5 KB


In [23]:
weather_daily['date'] = pd.to_datetime(weather_daily['date'])
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 365 non-null    datetime64[us]
 1   temperature_2m_max   365 non-null    float64       
 2   temperature_2m_min   365 non-null    float64       
 3   temperature_2m_mean  365 non-null    float64       
 4   precipitation_sum    365 non-null    float64       
 5   rain_sum             365 non-null    float64       
 6   snowfall_sum         365 non-null    float64       
 7   wind_speed_10m_max   365 non-null    float64       
dtypes: datetime64[us](1), float64(7)
memory usage: 22.9 KB


In [24]:
daily_rides.info()

<class 'pandas.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   date                   366 non-null    str  
 1   number_of_daily_rides  366 non-null    int64
dtypes: int64(1), str(1)
memory usage: 9.5 KB


In [25]:
daily_rides['date'] = pd.to_datetime(daily_rides['date'])
daily_rides.info()

<class 'pandas.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   366 non-null    datetime64[us]
 1   number_of_daily_rides  366 non-null    int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 5.8 KB


In [26]:
bike_weather_daily = daily_rides.merge(weather_daily,
                                       on='date',
                                       how='left')

bike_weather_daily.head()

,date,number_of_daily_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1174,11.0,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1709,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1764,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1336,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


## Daily Rides vs Average Temperature

In [27]:
fig = px.scatter(
    data_frame=bike_weather_daily,
    x='temperature_2m_mean',
    y='number_of_daily_rides',
    trendline='ols',
    title='Daily Rides vs Average Temperature'
)

fig.update_layout(
    xaxis_title='Average Daily temperature',
    yaxis_title='Number of Rides'
)

fig.show()

## Daily Rides vs Wind Speed

In [28]:
fig = px.scatter(
    data_frame=bike_weather_daily,
    x='wind_speed_10m_max',
    y='number_of_daily_rides',
    trendline='ols',
    title='Daily Rides vs Wind Speed'
)

fig.update_layout(
    xaxis_title='Wind Speed',
    yaxis_title='Number of Rides'
)

fig.show()

## Daily Rides vs Precipitation Sum

In [29]:
fig = px.scatter(
    data_frame=bike_weather_daily,
    x='precipitation_sum',
    y='number_of_daily_rides',
    trendline='ols',
    title='Daily Rides vs Precipitation Sum'
)

fig.update_layout(
    xaxis_title='Precipitation Sum',
    yaxis_title='Number of Rides'
)

fig.show()

## Dual Axis

In [30]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x = bike_weather_daily['date'],
        y = bike_weather_daily['number_of_daily_rides'],
        mode = 'lines',
        name = 'Daily Rides',
        yaxis = 'y1'
    )
)

fig.add_trace(
    go.Scatter(
        x = bike_weather_daily['date'],
        y = bike_weather_daily['temperature_2m_mean'],
        mode = 'lines',
        name = 'Daily Average Rides',
        yaxis = 'y2'
        )
)

fig.update_layout(
    title = '',
    xaxis = dict(title = 'Day'),
    yaxis = dict(title = 'Daily Rides',
                 side = 'left'),
    yaxis2 = dict(title = 'Temperature',
                 side = 'right', overlaying = 'y')
)

fig.show()